In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window 

df_stream = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/Volumes/fmcg/bronze/practice_volume/json_folder/schema")\
  .load("/Volumes/fmcg/bronze/practice_volume/json_folder")



In [0]:
## WRITE THE STREAMING DATA to a particular location instead of table 
try : 
    df_stream.writeStream\
            .option('checkpointLocation', '/Volumes/fmcg/bronze/practice_volume/json_folder/checkpoint2')\
            .trigger(availableNow=True)\
            .start('/Volumes/fmcg/bronze/practice_volume/json_folder/output')\
            .awaitTermination()  
except Exception as e: 
    print(f"Failed with error {str(e)}")   

In [0]:
# display(df_stream, checkpointLocation = "/Volumes/fmcg/bronze/practice_volume/json_folder/checkpoint")

## here I am getting error because I am not using any streaming aggregaration then spark streaming only supportes three modes , append, update ( updates existing rows when key changes ) and complete ( updates all rows) but in this case I am not using any streaming aggregaration so spark is not supporting complete mode and throwing error 


In [0]:
## write it to a table 

## error is coming because already the table is present in the proper location so thats why 
try : 
    df_stream.writeStream \
            .option('checkpointLocation', '/Volumes/fmcg/bronze/practice_volume/json_folder/checkpoint3')\
            .trigger(availableNow=True)\
            .table('fmcg.bronze.employee_details')\
            .awaitTermination()
except Exception as e :
       print(f"Failing with error {str(e)}")

In [0]:
df_emp = spark.sql("select * from fmcg.bronze.employee_details")
## per column how many nulls are there 
column_stat = []

null_counts = df_emp.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_emp.columns
]).collect()[0].asDict()

unique_counts = df_emp.select([
    approx_count_distinct(col(c)).alias(c) for c in df_emp.columns
]).collect()[0].asDict()


for col_nm, cl in enumerate(df_emp.columns[:-1],start=1) : 
    if ' ' in cl:
        print(f"bad column name {cl}")
        continue 
    column_stat.append(
        { 
         'colum_nmber': col_nm,
         'column_name' : cl,
          'data_type' : df_emp.schema[cl].dataType.typeName(),
          'null_count' : null_counts[cl],
          'unique_count' : unique_counts[cl]
        }
    )

col_stat = spark.createDataFrame(column_stat)
display(col_stat)


## we can see that in this case the the performace has been optimized by using dict method then looking up from ## that  so it got reduced to 3 sec

In [0]:
df_emp = spark.sql("select * from fmcg.bronze.employee_details")
## per column how many nulls are there 
column_stat_nw = []

for col_nm, cl in enumerate(df_emp.columns[:-1],start=1) : 
    if ' ' in cl:
        print(f"bad column name {cl}")
        continue 
    column_stat_nw.append(
        { 
         'colum_nmber': col_nm,
         'column_name' : cl,
          'data_type' : df_emp.schema[cl].dataType.typeName(),
          'null_count' : df_emp.filter(col(cl).isNull()).count(),
          'unique_count' : df_emp.select(cl).distinct().count()
        }
    )

col_stat_new = spark.createDataFrame(column_stat_nw)
display(col_stat_new)

## we have seen that it took around 11 sec to run the query so it is not optimized